# Step 5: Chain-level UniProt cofactor support for biological metals

This notebook keeps the original catalytic train CSV unchanged. It fetches chain-specific UniProt cofactor annotations through PDBe SIFTS + UniProt and writes auxiliary exact-PinMyMetal CSVs with chain-level annotation support columns.

Interpretation notes:

- What this can support: the mapped UniProt chain has a cofactor annotation mentioning the same normalized metal family as the row's `metaltype`.
- What this cannot prove: the specific predicted residue/site is experimentally native, geometrically correct, or the actual binding site for that cofactor. UniProt cofactor annotations are chain/protein-level evidence, not site-level validation.
- Missing or failed annotation is kept as unknown. Failed mapping, no chain-specific mapping, UniProt fetch failure, and no cofactor annotation become `site_native_status = unknown_annotation`; they are not treated as experimentally non-native.
- `chain_annotation_supports_metal = False` only means the available chain-level annotation did not list the row's normalized metal. It is not site-level disproof.
- These notebooks are hardcoded to `train_and_test_sets_structures_exact_pinmymetal`, which Plan.md treats as an auxiliary/possibly-overlapped exact split. The non-overlapped PinMyMetal split remains the trusted final held-out split.


In [ ]:
from pathlib import Path
import concurrent.futures
import re
import sys

import pandas as pd
import requests
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "DeepMzyme_Data").exists() and (candidate / "prepare_training_and_test_set").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
PREPARE_DIR = PROJECT_ROOT / "prepare_training_and_test_set"
if str(PREPARE_DIR) not in sys.path:
    sys.path.insert(0, str(PREPARE_DIR))

from structure_sync_utils import SUPPORTED_TRANSITION_METALS

DATASET_ROOT = PROJECT_ROOT / "DeepMzyme_Data" / "train_and_test_sets_structures_exact_pinmymetal"
TRAIN_DIR = DATASET_ROOT / "train"
TRAIN_CSV = TRAIN_DIR / "final_data_summarazing_table_transition_metals_only_catalytic.csv"
COFACTOR_ANALYSIS_CSV = TRAIN_DIR / "cofactor_analysis_by_structure_chain.csv"
WITH_CHAIN_ANNOTATION_STATUS_CSV = (
    TRAIN_DIR / "final_data_summarazing_table_transition_metals_only_catalytic_with_chain_annotation_status_exact_pinmymetal_auxiliary.csv"
)
CHAIN_ANNOTATION_SUPPORTED_CSV = (
    TRAIN_DIR / "final_data_summarazing_table_transition_metals_only_catalytic_chain_annotation_supported_metal_exact_pinmymetal_auxiliary.csv"
)

REQUEST_TIMEOUT = 20
MAX_WORKERS = 12
REQUIRED_COLUMNS = ["structure", "chain_resi", "metaltype", "ecnumber", "whether_catalytic"]
SUPPORTED_DATASET_METALS = set(SUPPORTED_TRANSITION_METALS)
METAL_NORMALIZATION = {
    "CO": "CO",
    "CU": "CU",
    "FE": "FE",
    "FE2": "FE",
    "FE3": "FE",
    "MG": "MG",
    "MN": "MN",
    "NI": "NI",
    "ZN": "ZN",
}
COFACTOR_SYMBOL_PATTERNS = {
    "CO": [r"^CO(?:\(\d\+\))?$", r"\bCO CATION\b", r"COBALT"],
    "CU": [r"^CU(?:\(\d\+\))?$", r"\bCU CATION\b", r"COPPER"],
    "FE": [r"^FE(?:\(\d\+\))?$", r"\bFE CATION\b", r"IRON"],
    "MG": [r"^MG(?:\(\d\+\))?$", r"\bMG CATION\b", r"MAGNESIUM"],
    "MN": [r"^MN(?:\(\d\+\))?$", r"\bMN CATION\b", r"MANGANESE"],
    "NI": [r"^NI(?:\(\d\+\))?$", r"\bNI CATION\b", r"NICKEL"],
    "ZN": [r"^ZN(?:\(\d\+\))?$", r"\bZN CATION\b", r"ZINC"],
}


def normalize_metal_symbol(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9]+", "", str(value).upper())
    return METAL_NORMALIZATION.get(cleaned, cleaned)


def extract_chain_id(chain_resi: str) -> str:
    chain_id, separator, _resseq = str(chain_resi).partition("_")
    if not separator or not chain_id.strip():
        raise ValueError(f"Could not parse chain from chain_resi={chain_resi!r}")
    return chain_id.strip()


def extract_protein_name(uniprot_json: dict) -> str:
    protein_description = uniprot_json.get("proteinDescription", {})
    recommended = protein_description.get("recommendedName")
    if isinstance(recommended, dict):
        full_name = recommended.get("fullName", {}).get("value")
        if full_name:
            return full_name
    for key in ("submissionNames", "alternativeNames"):
        for item in protein_description.get(key, []):
            full_name = item.get("fullName", {}).get("value")
            if full_name:
                return full_name
    return "Unknown"


def extract_annotated_metal_symbols(cofactor_names: list[str]) -> list[str]:
    symbols = set()
    for name in cofactor_names:
        upper_name = str(name).upper().strip()
        for symbol, patterns in COFACTOR_SYMBOL_PATTERNS.items():
            if any(re.search(pattern, upper_name) for pattern in patterns):
                symbols.add(symbol)
    return sorted(symbol for symbol in symbols if symbol in SUPPORTED_DATASET_METALS)


def choose_uniprot_for_chain(uniprot_entries: dict, chain_id: str) -> str | None:
    candidates = []
    for accession, info in uniprot_entries.items():
        matching_mappings = [
            mapping
            for mapping in info.get("mappings", [])
            if str(mapping.get("chain_id", "")).strip() == chain_id
        ]
        if not matching_mappings:
            continue
        best_coverage = max(float(mapping.get("coverage") or 0.0) for mapping in matching_mappings)
        best_identity = max(float(mapping.get("identity") or 0.0) for mapping in matching_mappings)
        candidates.append((best_coverage, best_identity, accession))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    return candidates[0][2]


def split_symbol_text(value: str) -> set[str]:
    if pd.isna(value) or not str(value).strip():
        return set()
    return {part.strip() for part in str(value).split(";") if part.strip()}


def annotation_status_is_unknown(value: str) -> bool:
    if pd.isna(value):
        return True
    return str(value).strip() != "ok"


def classify_chain_annotation_support(row: pd.Series) -> tuple[object, str]:
    if annotation_status_is_unknown(row.get("annotation_source_status")):
        return pd.NA, "unknown_annotation"

    annotated_symbols = split_symbol_text(row.get("Annotated Metal Symbols", ""))
    supports_metal = row["normalized_metaltype"] in annotated_symbols
    if supports_metal:
        return True, "supported_by_chain_annotation"
    return False, "unsupported_by_chain_annotation"


assert TRAIN_DIR.exists(), f"Train directory not found: {TRAIN_DIR}"
assert TRAIN_CSV.exists(), f"Train CSV not found: {TRAIN_CSV}"
print(f"Project root: {PROJECT_ROOT}")
print(f"Exact PinMyMetal train directory: {TRAIN_DIR}")
print("WARNING: exact PinMyMetal outputs are auxiliary/possibly overlapped; use the non-overlapped split for trusted final evaluation.")
print(f"Input CSV kept unchanged: {TRAIN_CSV}")
print(f"Cofactor analysis CSV: {COFACTOR_ANALYSIS_CSV}")
print(f"With chain-annotation-status CSV: {WITH_CHAIN_ANNOTATION_STATUS_CSV}")
print(f"Chain-annotation-supported CSV: {CHAIN_ANNOTATION_SUPPORTED_CSV}")


In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
missing_columns = [column for column in REQUIRED_COLUMNS if column not in df_train.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in {TRAIN_CSV}: {missing_columns}")

df_train["structure"] = df_train["structure"].astype(str).str.lower()
df_train["chain"] = df_train["chain_resi"].map(extract_chain_id)

targets = sorted({(row.structure, row.chain) for row in df_train[["structure", "chain"]].itertuples(index=False)})
print(f"Loaded {len(df_train)} train rows")
print(f"Fetching cofactor annotations for {len(targets)} unique structure/chain pairs")


def process_structure_chain(target: tuple[str, str]) -> dict[str, str]:
    structure, chain = target
    result = {
        "structure": structure,
        "chain": chain,
        "UniProt ID": "",
        "Protein Name": "Unknown",
        "Annotated Cofactors": "",
        "Annotated Metal Symbols": "",
        "Status": "ok",
    }

    try:
        sifts_url = f"https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{structure}"
        sifts_resp = requests.get(sifts_url, timeout=REQUEST_TIMEOUT)
        if sifts_resp.status_code != 200:
            result["Status"] = f"mapping_failed:{sifts_resp.status_code}"
            return result

        sifts_data = sifts_resp.json().get(structure, {})
        uniprot_entries = sifts_data.get("UniProt", {})
        if not uniprot_entries:
            result["Status"] = "no_uniprot_mapping"
            return result

        uniprot_id = choose_uniprot_for_chain(uniprot_entries, chain)
        if not uniprot_id:
            result["Status"] = "no_chain_specific_uniprot_mapping"
            return result

        result["UniProt ID"] = uniprot_id

        uniprot_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
        uniprot_resp = requests.get(uniprot_url, timeout=REQUEST_TIMEOUT)
        if uniprot_resp.status_code != 200:
            result["Status"] = f"uniprot_failed:{uniprot_resp.status_code}"
            return result

        uniprot_json = uniprot_resp.json()
        result["Protein Name"] = extract_protein_name(uniprot_json)

        cofactor_names = sorted(
            {
                cofactor.get("name", "").strip()
                for comment in uniprot_json.get("comments", [])
                if comment.get("commentType") == "COFACTOR"
                for cofactor in comment.get("cofactors", [])
                if cofactor.get("name")
            }
        )
        annotated_symbols = extract_annotated_metal_symbols(cofactor_names)

        result["Annotated Cofactors"] = "; ".join(cofactor_names)
        result["Annotated Metal Symbols"] = ";".join(annotated_symbols)
        if not cofactor_names:
            result["Status"] = "no_cofactor_annotation"

    except Exception as exc:
        result["Status"] = f"error:{type(exc).__name__}"

    return result


results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for result in executor.map(process_structure_chain, targets):
        results.append(result)

df_cofactor = pd.DataFrame(results).sort_values(["structure", "chain"]).reset_index(drop=True)
display(df_cofactor.head(20))
display(df_cofactor["Status"].value_counts(dropna=False).rename_axis("Status").reset_index(name="count"))

df_cofactor.to_csv(COFACTOR_ANALYSIS_CSV, index=False)
print(f"Saved cofactor analysis to {COFACTOR_ANALYSIS_CSV}")


In [ ]:
df_merged = df_train.merge(
    df_cofactor[["structure", "chain", "UniProt ID", "Protein Name", "Annotated Cofactors", "Annotated Metal Symbols", "Status"]],
    on=["structure", "chain"],
    how="left",
).rename(columns={"Status": "annotation_source_status"})

df_merged["normalized_metaltype"] = df_merged["metaltype"].map(normalize_metal_symbol)
support_status = df_merged.apply(classify_chain_annotation_support, axis=1, result_type="expand")
df_merged["chain_annotation_supports_metal"] = support_status[0]
df_merged["site_native_status"] = support_status[1]

status_counts = (
    df_merged["site_native_status"]
    .value_counts(dropna=False)
    .rename_axis("site_native_status")
    .reset_index(name="count")
)
display(status_counts)
display(
    df_merged[
        [
            "structure",
            "chain_resi",
            "metaltype",
            "normalized_metaltype",
            "ecnumber",
            "Annotated Cofactors",
            "Annotated Metal Symbols",
            "chain_annotation_supports_metal",
            "site_native_status",
            "annotation_source_status",
        ]
    ].head(20)
)

output_columns = REQUIRED_COLUMNS + [
    "normalized_metaltype",
    "chain_annotation_supports_metal",
    "site_native_status",
    "annotation_source_status",
    "UniProt ID",
    "Protein Name",
    "Annotated Cofactors",
    "Annotated Metal Symbols",
]
df_with_chain_status = df_merged[output_columns].copy()
unsupported_mask = ~df_with_chain_status["normalized_metaltype"].isin(SUPPORTED_DATASET_METALS)
unsupported_metaltypes = sorted(
    df_with_chain_status.loc[unsupported_mask, "normalized_metaltype"].dropna().unique()
)
if unsupported_metaltypes:
    print(f"Filtering unsupported normalized metal rows from exact train tables: {unsupported_metaltypes}")
df_with_chain_status = df_with_chain_status.loc[~unsupported_mask].copy()
df_with_chain_status["metaltype"] = df_with_chain_status["normalized_metaltype"]
supported_status_mask = df_with_chain_status["site_native_status"].eq("supported_by_chain_annotation")
df_chain_annotation_supported = df_with_chain_status[supported_status_mask].copy()

df_with_chain_status.to_csv(WITH_CHAIN_ANNOTATION_STATUS_CSV, index=False)
df_chain_annotation_supported.to_csv(CHAIN_ANNOTATION_SUPPORTED_CSV, index=False)

print(f"Original CSV left unchanged: {TRAIN_CSV}")
print(f"Saved chain-annotation-status CSV to {WITH_CHAIN_ANNOTATION_STATUS_CSV}")
print(f"Saved chain-annotation-supported CSV to {CHAIN_ANNOTATION_SUPPORTED_CSV}")
print(
    "Rows supported by chain-level annotation: "
    f"{len(df_chain_annotation_supported)} / {len(df_with_chain_status)}"
)
